# External PV challenge and EDGAR training audit
Run in a fresh Edge Colab runtime. Uses pinned HeartTwin scripts and measured murine pressure–volume recordings. Does not validate a human whole-heart model.
The observed PV run lost to the repeating training-waveform baseline. Preserve this result. The EDGAR training audit finds metadata, channel and geometry blockers; it executes no electrical solver. KIT fits, mouse 2/3 waveform values and EDGAR challenge testing data remain untouched.
This consolidated recipe is compiled; its constituent steps ran in Colab. It is not claimed to have been replayed end to end after consolidation.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time
ROOT=Path("/content")/("HeartTwin_external_challenge_"+time.strftime("%Y%m%d_%H%M%S"))
ROOT.mkdir()
repo=ROOT/"Virelion-HeartTwin"
subprocess.run(["git","clone","https://github.com/Virelion-Biotech/Virelion-HeartTwin.git",str(repo)],check=True)
subprocess.run(["git","-C",str(repo),"checkout","86579ea4f2e04a4059aa1a33d4716e705104d5c8"],check=True)
# Exact numerical versions of the observed run; use a fresh Colab runtime.
subprocess.run([sys.executable,"-m","pip","install","numpy==2.1.3","scipy==1.16.3","matplotlib==3.10.0","ruff"],check=True)
env=dict(os.environ,PYTHONPATH=str(repo),OPENBLAS_NUM_THREADS="1",MPLBACKEND="Agg")
with (ROOT/"challenge.log").open("w") as log:
    process=subprocess.run([sys.executable,str(repo/"scripts/challenge_external_pv.py"),"--output",str(ROOT/"results")],cwd=repo,env=env,stdout=log,stderr=subprocess.STDOUT)
print((ROOT/"challenge.log").read_text())
process.check_returncode()



In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from scipy.io import loadmat
from pathlib import Path
R=ROOT/"results"
models=json.loads((R/"frozen_models.json").read_text())
result=json.loads((R/"results.json").read_text())
data=loadmat(R/"m1_timeseries_final.mat")
pred=json.loads((R/"frozen_outer_predictions.json").read_text())
diagnosis={"purpose":"Post-score diagnosis only; no model or outer-score-selected refit.",
"source":"HeartTwin native chamber_pressure; no CardiFlow/CardiMech/CardiEP engine called",
"outer_predictions_frozen_before_pressure_scoring":True,"chambers":{}}
fig,axes=plt.subplots(2,2,figsize=(11,7),constrained_layout=True)
phase=np.linspace(0,1,50)
for row,ch in enumerate(["LV","RV"]):
    m=models[ch]; p=m["parameters"]; T=.1034
    V=data["V"+ch+"_stack"][:,40:50]/1000
    measured=data["P"+ch+"_stack"][:,40:50]
    elapsed=np.remainder(phase*T-p["activation_start_s"],T)
    duration=p["activation_duration_s"]; peak=duration*p.get("activation_peak_fraction",.5)
    a=np.where(elapsed<duration,np.sin(np.pi/2*np.where(elapsed<=peak,elapsed/peak,(duration-elapsed)/(duration-peak)))**2,0)
    delta=V-p["unstressed_volume_mL"]
    k=p.get("passive_exponent_per_mL",0.)
    passive=p["emin_mmHg_per_mL"]*delta if k==0 else p["emin_mmHg_per_mL"]*np.expm1(k*delta)/k
    reference=p["external_pressure_mmHg"]+(1-a[:,None])*passive+a[:,None]*p["emax_mmHg_per_mL"]*delta
    native=np.array(pred[ch])
    maxdiff=float(np.max(abs(reference-native)))
    assert maxdiff<1e-10
    selected=m["families"][m["selected_index"]]
    training=data["P"+ch+"_stack"][:,:40].mean(axis=1)
    diagnosis["chambers"][ch]={
      "independent_formula_max_difference_mmHg":maxdiff,
      "selected_stiffness_times_vref":selected["stiffness_times_vref"],
      "selected_asymmetric":selected["asymmetric"],
      "inner_rmse_all_families_mmHg":[f["inner_pressure_rmse_mmHg"] for f in m["families"]],
      "refit_active_mask":m["refit"]["active_mask"],
      "converged_trials":sum(t["success"] for f in m["families"] for t in f["calibration_trials"])+sum(t["success"] for t in m["refit_trials"]),
      "total_trials":sum(len(f["calibration_trials"]) for f in m["families"])+len(m["refit_trials"]),
      "pressure_range_mmHg":[float(measured.min()),float(measured.max())],
      "volume_range_mL":[float(V.min()),float(V.max())],
      "mean_error_by_phase_mmHg":(native-measured).mean(axis=1).tolist(),
      "diagnosis":"Formula agrees; remaining mismatch consistent with restricted phase/volume constitutive model and processed-beat variation. No unique physiological cause identified.",
      "baseline_comparison":"A repeating training waveform beats the fitted constitutive model on these processed beats; no predictive superiority established."}
    ax=axes[row,0]
    ax.plot(phase,measured,color=".75",alpha=.5)
    ax.plot(phase,measured.mean(axis=1),"k",label="Reserved-beat mean")
    ax.plot(phase,native.mean(axis=1),"C3",label="Native conditional pressure")
    ax.plot(phase,training,"C0",ls="--",label="Training waveform baseline")
    ax.set(xlabel="Publisher-normalized phase",ylabel=ch+" pressure (mmHg)")
    ax.legend(fontsize=8)
    ax=axes[row,1]
    ax.plot(phase,native-measured,alpha=.4,color="C3")
    ax.axhline(0,color="k",lw=.7)
    ax.set(xlabel="Publisher-normalized phase",ylabel=ch+" pressure residual (mmHg)")
fig.suptitle("External murine PV challenge: one animal, 10 reserved processed beats\nMeasured volume is an input; no circulation or human accuracy claim",fontsize=11)
fig.savefig(R/"external_pv_challenge.png",dpi=160)
(R/"diagnosis.json").write_text(json.dumps(diagnosis,indent=2,allow_nan=False))
print(json.dumps(diagnosis,indent=2))
display(fig)



In [ ]:
import subprocess, sys, json, hashlib, base64, zipfile
from pathlib import Path
subprocess.run([sys.executable,"-m","pip","install","-q","ruff"],check=True,capture_output=True)
checks={}
for name,cmd in {
"compile":[sys.executable,"-m","py_compile",str(repo/"scripts/challenge_external_pv.py")],
"lint":[sys.executable,"-m","ruff","check",str(repo/"scripts/challenge_external_pv.py")],
}.items():
    p=subprocess.run(cmd,cwd=repo,text=True,capture_output=True)
    checks[name]={"returncode":p.returncode,"stdout":p.stdout,"stderr":p.stderr}
(ROOT/"checks.json").write_text(json.dumps(checks,indent=2))
print(json.dumps(checks,indent=2))
print("CHALLENGE_EXIT",process.returncode)
print("RESERVED SUBJECT VALUES STILL UNOPENED")



In [ ]:
import json, subprocess
ED=ROOT/"edgar-2026-training"
with (ROOT/"edgar_training_contract.log").open("w") as log:
    result=subprocess.run([sys.executable,str(repo/"scripts/audit_edgar_2026_training.py"),"--output",str(ED)],cwd=repo,env=env,stdout=log,stderr=subprocess.STDOUT)
print((ROOT/"edgar_training_contract.log").read_text())
result.check_returncode()



In [ ]:
import zipfile,hashlib,json
archive=ROOT/"challenge_evidence.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file() and "Virelion-HeartTwin" not in p.parts and p.suffix in [".json",".log",".png"]:
            z.write(p,str(p.relative_to(ROOT)))
    for name in ["challenge_external_pv.py","audit_edgar_2026_training.py"]:
        z.write(repo/"scripts"/name,"executed_source/"+name)
print("Evidence archive:",archive)
print("SHA256",hashlib.sha256(archive.read_bytes()).hexdigest())
# Raw MAT files, EDGAR archives and coordinate/signal arrays are intentionally excluded.
from google.colab import files
files.download(str(archive))

